In [0]:
data1=[("ravi",1),("kumar",2)]
data2=[("ravi",1,1000),("kumar",2,2000)]

df1=spark.createDataFrame(data1,["name","id"])
df2=spark.createDataFrame(data2,["name","id","salary"])


#create delta table for data1
df1.write.format("delta").mode("overwrite").saveAsTable("scd1")
df1.select("name","id").show()

#load data2 using schema evolution technique
df2.write.format("delta").mode("append").option("mergeSchema","true").saveAsTable("scd1")
df2.select("name","id","salary").show()


In [0]:
#schema evolution using source as csv
df1=spark.read.format("csv").option("header","true").option("inferSchema","true").load("/Volumes/workspace/default/bronze/file2.csv.csv")
#df1.show()

#write into delta table scd2
df1.write.format("delta").mode("overwrite").saveAsTable("scd2")
df1.select("*").show()

#read 2nd file with new column added
df2=spark.read.format("csv").option("header","true").option("inferSchema","true").load("/Volumes/workspace/default/bronze/file1.csv.csv")



#update scd2 using schema evolution mode
df2.write.format("delta").mode("append").option("mergeSchema","true").saveAsTable("scd2")
df2.select("*").show()



In [0]:
# using explode
from pyspark.sql import SparkSession
from pyspark.sql.functions import split, explode, trim

spark = SparkSession.builder.appName("SportsExample").getOrCreate()

# Sample data
data = [
    ("alice", "Badminton,tennis"),
    ("Greg", "cricket,baseball"),
    ("Julie", "swimming,basket ball"),
    ("alan", "tennis,swimig"),
    ("Xian", "badminton,baseball")
]

df=spark.createDataFrame(data,["name","sports"])

df=df.select("name",split(trim("sports"),",").alias("sports"))
display(df)

# Split the 'sports' column into an array of individual sports
df_explode = df.withColumn("sports",explode("sports"))

# Show the result
df_explode.orderBy("sports","name").show()

In [0]:
%sql
-- merge into scd4 as target 
-- using scd3 as source 
-- on target.id=source.id 
-- when matched then update set target.name=source.name,target.department=source.department,target.salary=source.salary 
-- when not matched then insert(id,name,department,salary) values(source.id,source.name,source.department,source.salary)

select * from scd4



In [0]:
from pyspark.sql.functions import col,sum,max,count,row_number,rank,dense_rank
from pyspark.sql.window import Window

df = spark.table("scd3")
# df.show()
# df.select("*").filter(col("department")=='IT').show()
# df.groupBy("department").agg(count("*").alias("total")).filter(col("total")==2).orderBy(col("department")).show()
# df.groupBy("department").agg(max("salary")).show()

# using windows functions

df.withColumn("row_number",row_number().over(Window.partitionBy("department").orderBy(col("salary").desc()))).filter(col("row_number")==1).show()

#using sql


#find 2nd highest salary department wise
df.filter(col("salary").isNotNull()).withColumn("2nd_highest_salary",dense_rank().over(Window.partitionBy("department").orderBy(col("salary").desc()))).filter(col("2nd_highest_salary")==1).drop(col("2nd_highest_salary")).show()




In [0]:
from pyspark.sql.functions import col,min,max

data = [(1,), (3,), (5,), (7,), (9,), (10,)]
df = spark.createDataFrame(data, ["id"])
df.show()

# Create DataFrame with all numbers from min to max
# min_id = df.agg({"id": "min"}).collect()[0][0]
# max_id = df.agg({"id": "max"}).collect()[0][0]

min_id=df.select(min("id")).collect()[0][0]
max_id=df.select(max("id")).collect()[0][0]
print(min_id)
print(max_id)

all_nums = spark.range(min_id, max_id + 1).toDF("id")
# all_nums=spark.createDataFrame(all_nums,["id"])
all_nums.show()

# # Anti join to get missing numbers
missing_df = all_nums.join(df, on="id", how="left_anti")

missing_df.show()

In [0]:
for i in range(-1,10):
    print(i)

In [0]:
# data=[(a,x,200),(b,x,300),(c,x,400),(d,y,500),(e,z,600)]

data=[("a","x",200),("b","x",300),("c","x",400),("d","y",500),("e","z",600)]

df=spark.createDataFrame(data,["name","dept","salary"])
display(df)

In [0]:
#find employee name and max salary dept wise
from pyspark.sql.functions import col, max,count,dense_rank,row_number
from pyspark.sql.window import Window
df1 = df.groupBy("dept").agg(max(col("salary")).alias("max_salary"))
result = df.alias("e1").join(df1.alias("e2"), (col("e1.salary") == col("e2.max_salary")) & (col("e1.dept") == col("e2.dept")), "inner") \
           .select(col("e1.name"), col("e1.dept"), col("e2.max_salary"))
result.show()

# using window function
df.withColumn("max_salary",row_number().over(Window.partitionBy("dept").orderBy(col("salary").desc()))).filter(col("max_salary")==1).select("name","dept","salary").show()

           



In [0]:
#name of employee whose sal greater than two other employee from same dept

from pyspark.sql.functions import col, max,count,dense_rank
from pyspark.sql.window import Window
# result = (
#     df.groupBy("dept")
#       .agg(
#           count("*").alias("emp_cnt"),
#           max("salary").alias("max_salary")
#       )
#        .filter(col("emp_cnt") > 2)
#        .withColumnRenamed("dept","dept_no")
#       .join(
#           df.alias("e1"),
#           (col("e1.dept") == col("dept_no")) &
#           (col("e1.salary") == col("max_salary"))
#       )
#       .select(df.name, df.dept, df.salary)
# )

# result.show()

# using window functions
w_dept = Window.partitionBy("dept")
w_rank = Window.partitionBy("dept").orderBy(col("salary").desc())

result = (df
          .withColumn("emp_cnt", count("*").over(w_dept))
          .withColumn("rnk", dense_rank().over(w_rank))
            .filter((col("emp_cnt") > 2) & (col("rnk") == 1))
           .select("name", "dept", "salary"))

result.show()



